In [ ]:
# ============================================================
# TASK 7: TEXT SUMMARIZATION USING PRE-TRAINED MODEL (BART)
# Dataset: CNN-DailyMail (train.csv, validation.csv, test.csv)
# ============================================================

# ----------------------------
# 1️⃣ INSTALL REQUIRED PACKAGES
# ----------------------------
!pip install transformers datasets rouge-score evaluate pandas torch --quiet

# ----------------------------
# 2️⃣ IMPORT LIBRARIES
# ----------------------------
import pandas as pd
import torch
from transformers import BartTokenizer, BartForConditionalGeneration
from rouge_score import rouge_scorer
import evaluate
from tqdm import tqdm
import os

# ----------------------------
# 3️⃣ LOAD DATASET
# ----------------------------
# Upload your CSV files in Colab before running this

# Check if files exist before trying to load
required_files = ["train.csv", "validation.csv", "test.csv"]
for f in required_files:
    if not os.path.exists(f):
        raise FileNotFoundError(f"File not found: {f}. Please upload {f} to your Colab environment.")

try:
    train_df = pd.read_csv("train.csv")
    val_df = pd.read_csv("validation.csv")
    test_df = pd.read_csv("test.csv")
except FileNotFoundError as e:
    print(e)
    # Exit or handle the error appropriately if files are not found
    # For now, we'll re-raise to stop execution if files are missing
    raise

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# Check column names
print("Columns:", train_df.columns)

# CNN-DailyMail usually has columns:
# 'article' and 'highlights'
# If different, adjust column names below

# ----------------------------
# 4️⃣ LOAD PRE-TRAINED MODEL (BART)
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)

# ----------------------------
# 5️⃣ GENERATE SUMMARIES
# ----------------------------
def generate_summary(text):

    inputs = tokenizer(
        text,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=150,
        min_length=40,
        length_penalty=2.0,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# For faster evaluation, we take small subset
test_sample = test_df.head(100)

generated_summaries = []
reference_summaries = []

print("Generating summaries...")

for index, row in tqdm(test_sample.iterrows(), total=len(test_sample)):
    article = str(row["article"])
    reference = str(row["highlights"])

    summary = generate_summary(article)

    generated_summaries.append(summary)
    reference_summaries.append(reference)

# ----------------------------
# 6️⃣ EVALUATE USING ROUGE
# ----------------------------
print("\nCalculating ROUGE scores...")

rouge = evaluate.load("rouge")

results = rouge.compute(
    predictions=generated_summaries,
    references=reference_summaries
)

print("\nROUGE Scores:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

# ----------------------------
# 7️⃣ SHOW SAMPLE OUTPUT
# ----------------------------
print("\n================ SAMPLE RESULT ================")
print("Original Article:\n", test_sample.iloc[0]["article"])
print("\nReference Summary:\n", reference_summaries[0])
print("\nGenerated Summary:\n", generated_summaries[0])


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00
Train shape: (287113, 3)
Validation shape: (13368, 3)
Test shape: (11490, 3)
Columns: Index(['id', 'article', 'highlights'], dtype='object')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Generating summaries...


100%|██████████| 100/100 [59:57<00:00, 35.97s/it]



Calculating ROUGE scores...



ROUGE Scores:
rouge1: 0.4150
rouge2: 0.1945
rougeL: 0.2935
rougeLsum: 0.3549

================ SAMPLE RESULT ================
Original Article:
 Ever noticed how plane seats appear to be getting smaller and smaller? With increasing numbers of people taking to the skies, some experts are questioning if having such packed out planes is putting passengers at risk. They say that the shrinking space on aeroplanes is not only uncomfortable - it's putting our health and safety in danger. More than squabbling over the arm rest, shrinking space on planes putting our health and safety in danger? This week, a U.S consumer advisory group set up by the Department of Transportation said at a public hearing that while the government is happy to set standards for animals flying on planes, it doesn't stipulate a minimum amount of space for humans. 'In a world where animals have more rights to space and food than humans,' said Charlie Leocha, consumer representative on the committee. 'It is time that t